In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="FR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2548999786376953
epoch:  1, loss: 0.14582152664661407
epoch:  2, loss: 0.07254146784543991
epoch:  3, loss: 0.04725415259599686
epoch:  4, loss: 0.03748142346739769
epoch:  5, loss: 0.03401803597807884
epoch:  6, loss: 0.032780688256025314
epoch:  7, loss: 0.032340310513973236
epoch:  8, loss: 0.03217644244432449
epoch:  9, loss: 0.03210805356502533
epoch:  10, loss: 0.03207065165042877
epoch:  11, loss: 0.03181862458586693
epoch:  12, loss: 0.031721461564302444
epoch:  13, loss: 0.03166302293539047
epoch:  14, loss: 0.031645193696022034
epoch:  15, loss: 0.03163434565067291
epoch:  16, loss: 0.03159544616937637
epoch:  17, loss: 0.03155243396759033
epoch:  18, loss: 0.03153926134109497
epoch:  19, loss: 0.031515397131443024
epoch:  20, loss: 0.031488385051488876
epoch:  21, loss: 0.03145425766706467
epoch:  22, loss: 0.031442731618881226
epoch:  23, loss: 0.0314052514731884
epoch:  24, loss: 0.03138721361756325
epoch:  25, loss: 0.03132229298353195
epoch:  26, loss:

In [7]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7068569660186768
Test metrics:  R2 = 0.6076241731643677


In [8]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.29172271490097046
epoch:  1, loss: 0.15429863333702087
epoch:  2, loss: 0.10097673535346985
epoch:  3, loss: 0.06930141150951385
epoch:  4, loss: 0.05179162696003914
epoch:  5, loss: 0.04216746613383293
epoch:  6, loss: 0.036970920860767365
epoch:  7, loss: 0.034217435866594315
epoch:  8, loss: 0.032760366797447205
epoch:  9, loss: 0.03198539838194847
epoch:  10, loss: 0.03156742453575134
epoch:  11, loss: 0.03134017065167427
epoch:  12, loss: 0.031214537099003792
epoch:  13, loss: 0.031142372637987137
epoch:  14, loss: 0.031097806990146637
epoch:  15, loss: 0.031069805845618248
epoch:  16, loss: 0.030962485820055008
epoch:  17, loss: 0.030910514295101166
epoch:  18, loss: 0.030856903642416
epoch:  19, loss: 0.03081311658024788
epoch:  20, loss: 0.030720409005880356
epoch:  21, loss: 0.030615895986557007
epoch:  22, loss: 0.030507603660225868
epoch:  23, loss: 0.030397063121199608
epoch:  24, loss: 0.03036320209503174
epoch:  25, loss: 0.030259421095252037
epoch:  26

In [9]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6817072629928589
Test metrics:  R2 = 0.6038047075271606


In [10]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PRP+"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2420135885477066
epoch:  1, loss: 0.14475809037685394
epoch:  2, loss: 0.09196243435144424
epoch:  3, loss: 0.0634726732969284
epoch:  4, loss: 0.04832868650555611
epoch:  5, loss: 0.04040483012795448
epoch:  6, loss: 0.0363166369497776
epoch:  7, loss: 0.034231044352054596
epoch:  8, loss: 0.03317566588521004
epoch:  9, loss: 0.03264415264129639
epoch:  10, loss: 0.032376620918512344
epoch:  11, loss: 0.03224137797951698
epoch:  12, loss: 0.032172031700611115
epoch:  13, loss: 0.03213537484407425
epoch:  14, loss: 0.03211500868201256
epoch:  15, loss: 0.03210284933447838
epoch:  16, loss: 0.032083313912153244
epoch:  17, loss: 0.03203929215669632
epoch:  18, loss: 0.03199946880340576
epoch:  19, loss: 0.031985197216272354
epoch:  20, loss: 0.031947772949934006
epoch:  21, loss: 0.031905196607112885
epoch:  22, loss: 0.03187137097120285
epoch:  23, loss: 0.03179658204317093
epoch:  24, loss: 0.03174177557229996
epoch:  25, loss: 0.03171022608876228
epoch:  26, loss: 

In [11]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6875579357147217
Test metrics:  R2 = 0.6189771890640259


In [12]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="HS"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.32610419392585754
epoch:  1, loss: 0.1924198418855667
epoch:  2, loss: 0.044681258499622345
epoch:  3, loss: 0.044681258499622345
epoch:  4, loss: 0.0386057086288929
epoch:  5, loss: 0.03203259035944939
epoch:  6, loss: 0.03203259035944939
epoch:  7, loss: 0.03169017285108566
epoch:  8, loss: 0.03131985291838646
epoch:  9, loss: 0.03131985291838646
epoch:  10, loss: 0.031301338225603104
epoch:  11, loss: 0.03127678856253624
epoch:  12, loss: 0.03127678856253624
epoch:  13, loss: 0.03125004842877388
epoch:  14, loss: 0.031238669529557228
epoch:  15, loss: 0.0312128197401762
epoch:  16, loss: 0.031157035380601883
epoch:  17, loss: 0.03112226352095604
epoch:  18, loss: 0.031091684475541115
epoch:  19, loss: 0.03101472742855549
epoch:  20, loss: 0.03098117746412754
epoch:  21, loss: 0.030946264043450356
epoch:  22, loss: 0.030946264043450356
epoch:  23, loss: 0.030909989029169083
epoch:  24, loss: 0.030909989029169083
epoch:  25, loss: 0.03087269701063633
epoch:  26, los

In [13]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7127857208251953
Test metrics:  R2 = 0.6389390826225281


In [14]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="DY"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1267748773097992
epoch:  1, loss: 0.0835389494895935
epoch:  2, loss: 0.0835389494895935
epoch:  3, loss: 0.04122572019696236
epoch:  4, loss: 0.037067048251628876
epoch:  5, loss: 0.037067048251628876
epoch:  6, loss: 0.032787006348371506
epoch:  7, loss: 0.032335929572582245
epoch:  8, loss: 0.032335929572582245
epoch:  9, loss: 0.03183789178729057
epoch:  10, loss: 0.0317961759865284
epoch:  11, loss: 0.0317961722612381
epoch:  12, loss: 0.031746312975883484
epoch:  13, loss: 0.03173070773482323
epoch:  14, loss: 0.031729329377412796
epoch:  15, loss: 0.03172755986452103
epoch:  16, loss: 0.031714413315057755
epoch:  17, loss: 0.031714413315057755
epoch:  18, loss: 0.031698428094387054
epoch:  19, loss: 0.03169843181967735
epoch:  20, loss: 0.03169843181967735
epoch:  21, loss: 0.03168881684541702
epoch:  22, loss: 0.031688056886196136
epoch:  23, loss: 0.03167511522769928
epoch:  24, loss: 0.03167511522769928
epoch:  25, loss: 0.0316595658659935
epoch:  26, loss:

In [15]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8845587968826294
Test metrics:  R2 = 0.8547267317771912
